In [1]:
# ==========================================
# Imports
# ==========================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from anova_module import FullSupportAnova, batch_shapley_values
import shap

In [2]:
# ==========================================
# 1. Data Acquisition and Preprocessing
# ==========================================
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/nursery/nursery.data"

# Load dataset directly (no header provided in source file)
# Features: parents, has_nurs, form, children, housing, finance, social, health, class
df = pd.read_csv(url, header=None)

# Split features (X) and target (y)
X_raw = df.iloc[:, :-1]  # First 8 columns (features)
y_raw = df.iloc[:, -1]   # Last column (target class)

# Encode categorical features to integers (Ordinal Encoding)
# Note: Using LabelEncoder instead of OneHot to maintain the specific (N, 8) input dimensionality constraint
le = LabelEncoder()
X_encoded = X_raw.apply(le.fit_transform)
y_encoded = le.fit_transform(y_raw)

# Convert to PyTorch tensors
X = torch.tensor(X_encoded.values, dtype=torch.float32) # nn.Linear requires Float32 input
y = torch.tensor(y_encoded, dtype=torch.long)           # CrossEntropyLoss requires Long (int64) targets

print(f"Dataset downloaded.")
print(f"X matrix shape: {X.shape}")  # Should display torch.Size([12960, 8])

# Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. PyTorch MLP Architecture Definition
# ==========================================
class NurseryMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NurseryMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.network(x)

# Model Initialization
input_dim = X.shape[1]           # 8 input features
output_dim = len(set(y_encoded)) # 5 output classes
model = NurseryMLP(input_dim, output_dim)

# Loss and Optimizer configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

# ==========================================
# 3. Training Loop
# ==========================================
epochs = 30
batch_size = 64
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print("\nStarting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

# ==========================================
# 4. Evaluation (Accuracy)
# ==========================================
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)

print(f"\nTest set accuracy: {accuracy*100:.2f}%")

Dataset downloaded.
X matrix shape: torch.Size([12960, 8])

Starting training...
Epoch 10/30 - Loss: 0.0672
Epoch 20/30 - Loss: 0.0319
Epoch 30/30 - Loss: 0.0204

Test set accuracy: 99.73%


In [ ]:
%%time
# ==========================================
# Functional ANOVA Decomposition
# ==========================================

X = X_encoded.to_numpy() # Dataset
d = X.shape[1] # dimension
N = [ X[: , j].max() + 1 for j in range(d) ] # list of categories
r = np.prod(N) # full dimension
P = 1/r * np.ones( r ) # vector of probabilities

def f(i, X_numpy): # Proba( MLP(x) = class i | x )

    tensor_input = torch.tensor(X_numpy, dtype=torch.float32)
    
    model.eval()
    with torch.no_grad():
        
        logits = model(tensor_input)      
        
        probs = torch.softmax(logits, dim=1)
        
        predictions_class_i = probs[:, i]

    return predictions_class_i.cpu().numpy()

def f_0(x): # we focus only on class 0
    return(f(0 , x))

def f_1(x): # we focus only on class 1
    return(f(1 , x))

def f_2(x): # we focus only on class 2
    return(f(2 , x))

def f_3(x): # we focus only on class 3
    return(f(3 , x))

def f_4(x): # we focus only on class 4
    return(f(4 , x))

F = [f_0 , f_1 , f_2 , f_3 , f_4] # list of functions for each class

anova_shap = [] # list of generalized shapley values based on functional anova
for f_model in F:
    A = FullSupportAnova(N , P , f_model)
    S , Matrix = A.get_anova_full() # sets and f_A(X_A)
    shap_i = batch_shapley_values(d , S , Matrix) # generalized shapley values matrix for all obs
    anova_shap.append(shap_i)

CPU times: user 2min 37s, sys: 4.94 s, total: 2min 42s
Wall time: 53.7 s


In [13]:
# ==========================================
# KernelSHAP
# ==========================================

n_sample_background = 200
background = X[:n_sample_background]

# KernelSHAP
def kernel_shap(f , X_explain):
    explainer = shap.KernelExplainer(f , background)
    shap_values = explainer.shap_values(X_explain)
    return(shap_values)

In [ ]:
%%time
# ==========================================
# KernelSHAP
# ==========================================

number = X.shape[0] - 1
X_explain = A._generate_tuples()[:number]

all_kernel_shap = []
for g in F:
    kernel_shap_g = kernel_shap(g , X_explain)
    all_kernel_shap.append(kernel_shap_g)

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/12959 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/12959 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


...

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/12959 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/12959 [00:00<?, ?it/s]

CPU times: user 7min 12s, sys: 3min 8s, total: 10min 21s
Wall time: 5min 6s


In [15]:
# ==========================================
# Table of MSE
# ==========================================

P_red = 1/number * np.ones(number)

np.array([np.sum(((anova_shap[i][:number , :] - all_kernel_shap[i])**2).T * P_red , axis=1) for i in range(5)])

array([[3.04847562e-10, 2.40328281e-10, 2.22418924e-10, 7.28466824e-11,
        2.75569893e-11, 1.40257661e-11, 4.72785170e-11, 1.11166186e-05],
       [1.63788959e-02, 3.69209523e-02, 2.93930668e-03, 2.80880975e-03,
        1.71145172e-03, 3.52273307e-04, 3.21433367e-03, 1.78844183e-02],
       [2.61574330e-14, 2.95245237e-14, 2.15717993e-14, 3.42115072e-14,
        1.18681794e-14, 1.07816684e-14, 1.90154506e-14, 2.53373432e-14],
       [1.88924993e-02, 4.42082090e-02, 2.06204039e-03, 4.52357028e-04,
        8.21612019e-04, 2.35157069e-04, 7.31613104e-04, 1.67232584e-02],
       [2.48433624e-03, 2.84381425e-03, 1.42883943e-03, 1.33636015e-03,
        2.56063193e-04, 2.38811581e-05, 9.46781525e-04, 3.20483052e-03]])